# MASLD-LLM · 03 — Fine-tuning on Colab (Drive-synced edition)



Trains the model ladder from your Drive-synced folders. Sync mounts at

`/content/drive/Othercomputers/My Mac/` (read-only); checkpoints go to the

writable fallback if needed.



## Session 5 (smoke test): runtime = **L4**, model = **Qwen3-0.6B**



Paper recipe: LoRA r32 α32, ChatML, 3 epochs, assistant-only loss, neat packing.


## 0 · Mount + auto-discover


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']

def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None:
        search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct):
            matches.append(direct)
    if matches:
        return matches
    def walk(path, d):
        if d > depth:
            return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name:
                        matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError):
            pass
    for root in search_roots:
        if os.path.exists(root):
            walk(root, 0)
    return matches

cands = find_dir('masld-llm')
MASLD_LLM = min(cands, key=len) if cands else None
assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive'
PROMPTS = os.path.join(MASLD_LLM, 'prompts')
print('PROMPTS =', PROMPTS)
print('corpora:', sorted(f for f in os.listdir(PROMPTS) if f.endswith('.jsonl.gz')))

def is_writable(path):
    try:
        probe = os.path.join(path, '.write_test')
        open(probe, 'w').close(); os.remove(probe)
        return True
    except Exception:
        return False

CKPT = (os.path.join(MASLD_LLM, 'checkpoints') if is_writable(MASLD_LLM)
        else '/content/drive/MyDrive/masld-llm-outputs/checkpoints')
os.makedirs(CKPT, exist_ok=True)
print('CKPT (writable) =', CKPT)

## 1 · Install + convert all seven train corpora


In [ ]:
%cd /content
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip -q install -e ".[torch,metrics]" bitsandbytes peft transformers accelerate

import gzip, json, os

def to_alpaca(in_path, out_path):
    rows = []
    with gzip.open(in_path, 'rt') as f:
        first = json.loads(f.readline())
        if '__meta__' not in first:
            rows.append(first)
        for l in f:
            rows.append(json.loads(l))
    out = [dict(instruction=r['prompt'], output=str(r['answer'])) for r in rows]
    json.dump(out, open(out_path, 'w'), indent=0)
    print(out_path, len(out))

to_alpaca(f'{PROMPTS}/MB-10xx_liver_train.jsonl.gz',      '/content/sft_liver.json')
to_alpaca(f'{PROMPTS}/MB-11xx_external_train.jsonl.gz',   '/content/sft_external.json')
to_alpaca(f'{PROMPTS}/MB-61xx_msigdb_train.jsonl.gz',     '/content/sft_knowledge.json')
to_alpaca(f'{PROMPTS}/MB-62xx_drugs_train.jsonl.gz',      '/content/sft_drugs.json')
to_alpaca(f'{PROMPTS}/MB-63xx_drugcentral_affinity_train.jsonl.gz', '/content/sft_drugs2.json')
to_alpaca(f'{PROMPTS}/MB-70xx_dge_direction_train.jsonl.gz', '/content/sft_dge.json')
to_alpaca(f'{PROMPTS}/MB-8xxx_fujiwara_train.jsonl.gz',   '/content/sft_fujiwara.json')

In [ ]:
# hierarchical-uniform domain sampling -> ~60k-prompt mixture
import json, os, random
rs = random.Random(20260929)

DOMAIN_SHARE = {
  'liver':    (['/content/sft_liver.json', '/content/sft_external.json', '/content/sft_fujiwara.json'], 0.35),
  'knowledge':(['/content/sft_knowledge.json', '/content/sft_dge.json'], 0.15),
  'drugs':    (['/content/sft_drugs.json', '/content/sft_drugs2.json'], 0.25),
  'dnam':     (['/content/sft_dnam.json'], 0.05),
  'tahoe':    (['/content/sft_tahoe.json'], 0.20),
}
TARGET = 60_000

pool = {}
for k, (fps, share) in DOMAIN_SHARE.items():
    rows = []
    for fp in fps:
        if os.path.exists(fp):
            rows += json.load(open(fp))
    pool[k] = rows
    print(k, len(rows))

out = []
for k, (fps, share) in DOMAIN_SHARE.items():
    rows = pool.get(k) or []
    n = int(TARGET * share)
    if len(rows) > n:
        rows = rs.sample(rows, n)
    out += rows
rs.shuffle(out)
json.dump(out, open('/content/sft_multitask.json', 'w'), indent=0)
print('mixture size:', len(out))

## 2 · Register dataset + 0.6B config


In [ ]:
ds_entry = '''{
  "masld_sft": {
    "file_name": "/content/sft_multitask.json",
    "columns": {"prompt": "instruction", "response": "output"}
  }
}'''
open('data/dataset_info.json', 'w').write(ds_entry)

yaml_06b = '''
model_name_or_path: Qwen/Qwen3-0.6B
trust_remote_code: true
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_alpha: 32
lora_dropout: 0.0
dataset: masld_sft
template: qwen
cutoff_len: 8192
packing: true
neat_packing: true
max_samples: 100000
overwrite_cache: true
preprocessing_num_workers: 16
output_dir: ''' + CKPT + '''/qwen3-0.6b-masld-sft
logging_steps: 10
save_steps: 0.25
plot_loss: true
per_device_train_batch_size: 2
gradient_accumulation_steps: 16
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
'''
open('/content/qwen06b_sft.yaml', 'w').write(yaml_06b)
print('config written; CKPT =', CKPT)

In [ ]:
# Session 5 smoke test: watch loss drop 4-5 -> <2 within 50 steps
!llamafactory-cli train /content/qwen06b_sft.yaml

## 3 · Evaluate on the frozen test (first C2-contrast row)


In [ ]:
!pip -q install vllm
from vllm import LLM, SamplingParams
import gzip, json, random, sys, os

# merge adapter first via subprocess (shell escape kept minimal)
import subprocess
subprocess.run(['llamafactory-cli', 'export', 'merge',
                '--model_name_or_path', 'Qwen/Qwen3-0.6B',
                '--adapter_name_or_path', f'{CKPT}/qwen3-0.6b-masld-sft',
                '--export_dir', f'{CKPT}/qwen3-0.6b-masld-merged'], check=True)

llm = LLM(model=f'{CKPT}/qwen3-0.6b-masld-merged', gpu_memory_utilization=0.9)
sp = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=200)

rng = random.Random(20260929)
rows = []
with gzip.open(f'{PROMPTS}/MB-20xx_liver_test.jsonl.gz', 'rt') as f:
    f.readline()
    for line in f:
        rows.append(json.loads(line))
by_task = {}
for r in rows:
    by_task.setdefault(r['task'], []).append(r)
sample = []
for task, items in sorted(by_task.items()):
    rng.shuffle(items)
    sample.extend(items[:300])

outs = llm.generate([r['prompt'] for r in sample], sp)

sys.path.insert(0, '/content')
from session3_frontier import parse_answer
runs_dir = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
            if os.access(os.path.join(MASLD_LLM, 'results'), os.W_OK)
            else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(runs_dir, exist_ok=True)
run_path = runs_dir + '/qwen3-0.6b-masld__MB-20xx.jsonl.gz'
with gzip.open(run_path, 'wt') as f:
    for r, o in zip(sample, outs):
        raw = o.outputs[0].text
        f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                raw=raw, error=None, model_version='qwen3-0.6b-masld-sft',
                                parsed=parse_answer(raw, r['task']))) + chr(10))
print('archived ->', run_path)

## 4 · Ladder continuation

- **1.7B**: same config, `model_name_or_path: Qwen/Qwen3-1.7B`.

- **9B two-stage QLoRA**: A100 40GB (Pro+). Stage A = KG/drug text, 1 epoch, lr 2e-5;

  stage B = this config from the stage-A adapter, lr 1e-4, `quantization_bit: 4`.

- After each run: sync home, then locally `python3 scripts/score_frontier.py`.
